In [ ]:
# Skip restarting message in Colab
import sys; modules = list(sys.modules.keys())
for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None

# 1. Install vLLM first (0.19.1 is a solid choice for Colab)
!pip install vllm==0.19.1

# 2. Install Unsloth with its specific Colab dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. Install the "Big Three" for GRPO with strict versions
# Note: We put trl==0.14.0 here to ensure it isn't overwritten
!pip install --no-deps trl==0.14.0 peft accelerate bitsandbytes xformers

# 4. Install supporting utilities
!pip install --no-deps unsloth_zoo torchcodec
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps diffusers

# 5. Fix Transformers to a version compatible with Unsloth's current hooks
!pip install --upgrade "transformers>=4.48.0" 
!pip install --upgrade pillow

In [2]:
import os, importlib.util
!pip install --upgrade uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install \
        "torch>=2.9.1" "torchcodec==0.9.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        "vllm==0.16.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install unsloth
# Gemma 4 requires transformers >= 5.5.0
!uv pip install --upgrade --no-deps "transformers>=5.5.0" "tokenizers>=0.22.0,<=0.23.0" "trl>=0.28.0" "huggingface-hub>=1.5.0,<2.0" unsloth unsloth_zoo

Using Python 3.12.13 environment at: /usr
Resolved 193 packages in 1.53s                                       
Prepared 1 package in 240ms                                              
Uninstalled 4 packages in 93ms
Installed 4 packages in 62ms                                
 - huggingface-hub==1.13.0
 + huggingface-hub==0.36.2
 - torchcodec==0.10.0+cu128
 + torchcodec==0.9.0
 - transformers==5.7.0
 + transformers==4.57.6
 - trl==1.3.0
 + trl==0.24.0
Using Python 3.12.13 environment at: /usr
Resolved 6 packages in 159ms                                         
Prepared 3 packages in 0.46ms                                            
Uninstalled 3 packages in 86ms
Installed 3 packages in 61ms                                
 - huggingface-hub==0.36.2
 + huggingface-hub==1.13.0
 - transformers==4.57.6
 + transformers==5.7.0
 - trl==0.24.0
 + trl==1.3.0


In [2]:
import torch
print(f"Torch version: {torch.__version__}")
import torchcodec
print("Torchcodec loaded successfully!")

from unsloth import FastLanguageModel, PatchFastRL
#PatchFastRL("GRPO", FastLanguageModel)

Torch version: 2.9.1+cu128
Torchcodec loaded successfully!
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Model Loading

In [ ]:
from unsloth import FastModel
import torch

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model_name = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit"

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    dtype = None, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
temperature = 0.8

from vllm import SamplingParams
import random
import io
from collections import Counter

sampling_params = SamplingParams(
    temperature = temperature,
    top_p = 0.95,
    max_tokens = 512,
)